In [1]:
from datasets import load_dataset
import os

os.environ['http_proxy'] = 'http://127.0.0.1:10809'
os.environ['https_proxy'] = 'http://127.0.0.1:10809'

In [5]:

data = load_dataset('TAAC2026/data_sample_1000', cache_dir='.cache')

README.md: 0.00B [00:00, ?B/s]

sample_data.parquet:   0%|          | 0.00/71.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [6]:
data

DatasetDict({
    train: Dataset({
        features: ['item_id', 'item_feature', 'label', 'seq_feature', 'timestamp', 'user_feature', 'user_id'],
        num_rows: 1000
    })
})

In [7]:
data['train']

Dataset({
    features: ['item_id', 'item_feature', 'label', 'seq_feature', 'timestamp', 'user_feature', 'user_id'],
    num_rows: 1000
})

In [9]:
train_data = data['train']

In [10]:
# 打乱
train_data.shuffle()

Dataset({
    features: ['item_id', 'item_feature', 'label', 'seq_feature', 'timestamp', 'user_feature', 'user_id'],
    num_rows: 1000
})

In [11]:
# 重命名字段
train_data.rename_column('item_id', 'id')

Dataset({
    features: ['id', 'item_feature', 'label', 'seq_feature', 'timestamp', 'user_feature', 'user_id'],
    num_rows: 1000
})

In [14]:
# 数据处理
def map_fun(sample):
    sample['user_id'] = 'map' + sample['user_id']
    return sample


train_data.map(map_fun)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset({
    features: ['item_id', 'item_feature', 'label', 'seq_feature', 'timestamp', 'user_feature', 'user_id'],
    num_rows: 1000
})

In [16]:
# 批量并发处理
def map_fun_batch(sample_batch):
    user_id_batch = sample_batch['user_id']
    user_id_batch = [f'map_batch_{user_id}' for user_id in user_id_batch]
    sample_batch['user_id'] = user_id_batch
    return sample_batch


train_data.map(
    function=map_fun_batch,
    batched=True,
    batch_size=100,
    num_proc=4
)

Map (num_proc=4):   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset({
    features: ['item_id', 'item_feature', 'label', 'seq_feature', 'timestamp', 'user_feature', 'user_id'],
    num_rows: 1000
})

In [17]:
# 设置数据格式
train_data.set_format(type='torch', columns=['timestamp'], output_all_columns=True)

In [20]:
train_data[10]['timestamp']

tensor(1770700166)

# 3 保存数据为其他格式

In [21]:
# 保存成CSV
train_data.to_csv(path_or_buf='./data/data_sample_1000.csv')

Creating CSV from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

76362289

In [22]:
csv_dataset = load_dataset(path='csv', data_files='./data/data_sample_1000.csv',split='train')
csv_dataset


Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['item_id', 'item_feature', 'label', 'seq_feature', 'timestamp', 'user_feature', 'user_id'],
    num_rows: 1000
})